# SeaweedFS, Polaris and history
Run beside the matching Markdown runbook. Start with a fresh Python kernel. Keep the Spark session alive until the inspection checkpoint. Configuration is read from config.json beside this notebook.


In [ ]:
import json, os, socket, uuid
from pathlib import Path
from pyspark.sql import SparkSession
config = json.loads(Path('config.json').read_text())
required = config['credential_environment']
missing = [name for name in required.values() if not os.environ.get(name)]
assert not missing, 'Missing configured credential variables: ' + ', '.join(missing)
credentials = {key: os.environ[name] for key, name in required.items()}
run_id = 'ops_' + uuid.uuid4().hex[:12]
root = f"s3a://{config['bucket']}/{config['temporary_prefix']}/{run_id}"
print('Run:', run_id, 'Temporary data:', root)
# Values remain in memory; never display credentials or the full Spark configuration.


In [ ]:
builder = (SparkSession.builder.appName(run_id)
    .master(config['spark_master'])
    .config('spark.driver.host', os.environ.get('POD_IP') or socket.gethostbyname(socket.gethostname()))
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.ui.port', '4040')
    .config('spark.cores.max', '2')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.eventLog.enabled', 'true')
    .config('spark.eventLog.dir', config['event_log_directory'])
    .config('spark.redaction.regex', '(?i)secret|password|token|access.?key|credential')
    .config('spark.hadoop.fs.s3a.endpoint', config['s3_endpoint'])
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.access.key', credentials['s3_access_key'])
    .config('spark.hadoop.fs.s3a.secret.key', credentials['s3_secret_key'])
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', config['polaris_uri'])
    .config('spark.sql.catalog.polaris.warehouse', config['polaris_warehouse'])
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.credential', credentials['polaris_client_id'] + ':' + credentials['polaris_client_secret'])
    .config('spark.sql.catalog.polaris.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.polaris.s3.endpoint', config['s3_endpoint'])
    .config('spark.sql.catalog.polaris.s3.path-style-access', 'true')
    .config('spark.sql.catalog.polaris.s3.access-key-id', credentials['s3_access_key'])
    .config('spark.sql.catalog.polaris.s3.secret-access-key', credentials['s3_secret_key'])
    .config('spark.sql.catalog.polaris.client.region', 'us-east-1'))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.master == config['spark_master'], 'Restart this kernel: an old Spark context was reused'
application_id = spark.sparkContext.applicationId
print('Application:', application_id, 'Driver UI:', spark.sparkContext.uiWebUrl)


In [ ]:
orders = spark.createDataFrame([(1,120),(2,80),(3,50),(4,150)], 'id long, amount long')
orders.write.parquet(root + '/orders')
assert spark.read.parquet(root + '/orders').orderBy('id').collect() == orders.orderBy('id').collect()
spark.sql(f'CREATE NAMESPACE polaris.{run_id}')
orders.writeTo(f'polaris.{run_id}.orders').using('iceberg').create()
assert spark.table(f'polaris.{run_id}.orders').orderBy('id').collect() == orders.orderBy('id').collect()
spark.sql(f'UPDATE polaris.{run_id}.orders SET amount = 100 WHERE id = 2')
assert spark.table(f'polaris.{run_id}.orders').groupBy().sum('amount').first()[0] == 420
snapshots = spark.sql(f'SELECT snapshot_id, operation FROM polaris.{run_id}.orders.snapshots')
snapshots.show(truncate=False)
assert snapshots.count() >= 2
print('PASS: S3 round trip, Polaris read/write/update, total 420 and multiple snapshots')


In [ ]:
location = next(r.data_type for r in spark.sql(f'DESCRIBE EXTENDED polaris.{run_id}.orders').collect() if r.col_name == 'Location')
uri = spark._jvm.java.net.URI(location.replace('s3://','s3a://',1))
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(uri, spark._jsc.hadoopConfiguration())
files = fs.listFiles(spark._jvm.org.apache.hadoop.fs.Path(str(uri)), True)
keys = []
while files.hasNext():
    keys.append(str(files.next().getPath()))
assert any(k.endswith('.parquet') for k in keys)
assert any(k.endswith('.metadata.json') for k in keys)
print('PASS: physical Parquet data and Iceberg metadata objects:', len(keys))
print('Polaris table location:', location)


Checkpoint: inspect logs and the live application. The plain Parquet prefix and the catalog-managed Iceberg location are separate. Save both if retaining the exercise.


In [ ]:
# Remove only this run's catalog objects. Plain Parquet remains for inspection.
spark.sql(f'DROP TABLE polaris.{run_id}.orders PURGE')
spark.sql(f'DROP NAMESPACE polaris.{run_id}')
print('PASS: temporary Polaris table and namespace removed')


In [ ]:
record = {'run_id': run_id, 'application_id': application_id, 'root': root}
Path(run_id + '.json').write_text(json.dumps(record, indent=2))
spark.stop()
print('PASS: session stopped; inspect the completed application in Spark History Server')
